# Evaluating Multiple LM Outputs (External)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code
from stat_genie.blade_pipeline.additions.eval.judge import \
    run_judge_evaluation_pairwise
from blade_bench.utils import get_dataset_info_path, get_dataset_csv_path

In [4]:
# load files
analysis_subdir_path_1 = "outputs/analysis1_output"
analysis_subdir_path_2 = "outputs/analysis2_output"
analysis_subdir_path_3 = "outputs/analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [5]:
# load dataset info and csv to get task and dataframe
info_path = get_dataset_info_path(multirun_analyses_1["dataset_name"])
data_path = get_dataset_csv_path(multirun_analyses_1["dataset_name"])

with open(info_path, "r") as file:
    info_json = json.load(file)
    
dataset_task = info_json["research_questions"][0]
df = pd.read_csv(data_path)

In [6]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)


[2025-12-17 09:26:59.30][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/campus/austin.zane/stat-genie/config/llm_config.yml'.


In [7]:
# Load or generate features for each analysis group
features_file_1 = join(analysis_subdir_path_1, "features.json")
features_file_2 = join(analysis_subdir_path_2, "features.json")
features_file_3 = join(analysis_subdir_path_3, "features.json")

if os.path.exists(features_file_1):
    print("Loading features_1 from file...")
    with open(features_file_1, "r", encoding="utf-8") as f:
        features_1 = json.load(f)
    # Convert string keys back to int keys
    features_1 = {int(k): v for k, v in features_1.items()}
else:
    print("Generating features_1 (this will make LLM API calls)...")
    features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
    # Save to file (convert int keys to strings for JSON)
    with open(features_file_1, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in features_1.items()}, f, indent=2)
    print(f"Saved features_1 to {features_file_1}")

if os.path.exists(features_file_2):
    print("Loading features_2 from file...")
    with open(features_file_2, "r", encoding="utf-8") as f:
        features_2 = json.load(f)
    features_2 = {int(k): v for k, v in features_2.items()}
else:
    print("Generating features_2 (this will make LLM API calls)...")
    features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
    with open(features_file_2, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in features_2.items()}, f, indent=2)
    print(f"Saved features_2 to {features_file_2}")

if os.path.exists(features_file_3):
    print("Loading features_3 from file...")
    with open(features_file_3, "r", encoding="utf-8") as f:
        features_3 = json.load(f)
    features_3 = {int(k): v for k, v in features_3.items()}
else:
    print("Generating features_3 (this will make LLM API calls)...")
    features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)
    with open(features_file_3, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in features_3.items()}, f, indent=2)
    print(f"Saved features_3 to {features_file_3}")

Loading features_1 from file...
Loading features_2 from file...
Loading features_3 from file...


In [8]:
# Load or generate model_info for each analysis group
model_info_file_1 = join(analysis_subdir_path_1, "model_info.json")
model_info_file_2 = join(analysis_subdir_path_2, "model_info.json")
model_info_file_3 = join(analysis_subdir_path_3, "model_info.json")

if os.path.exists(model_info_file_1):
    print("Loading model_info_1 from file...")
    with open(model_info_file_1, "r", encoding="utf-8") as f:
        model_info_1_dict = json.load(f)
    # Convert string keys back to int keys
    model_info_1 = {int(k): v for k, v in model_info_1_dict.items()}
else:
    print("Generating model_info_1 (this will make LLM API calls)...")
    model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
    # Save to file (convert int keys to strings for JSON)
    with open(model_info_file_1, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in model_info_1.items()}, f, indent=2)
    print(f"Saved model_info_1 to {model_info_file_1}")

if os.path.exists(model_info_file_2):
    print("Loading model_info_2 from file...")
    with open(model_info_file_2, "r", encoding="utf-8") as f:
        model_info_2_dict = json.load(f)
    model_info_2 = {int(k): v for k, v in model_info_2_dict.items()}
else:
    print("Generating model_info_2 (this will make LLM API calls)...")
    model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
    with open(model_info_file_2, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in model_info_2.items()}, f, indent=2)
    print(f"Saved model_info_2 to {model_info_file_2}")

if os.path.exists(model_info_file_3):
    print("Loading model_info_3 from file...")
    with open(model_info_file_3, "r", encoding="utf-8") as f:
        model_info_3_dict = json.load(f)
    model_info_3 = {int(k): v for k, v in model_info_3_dict.items()}
else:
    print("Generating model_info_3 (this will make LLM API calls)...")
    model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)
    with open(model_info_file_3, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in model_info_3.items()}, f, indent=2)
    print(f"Saved model_info_3 to {model_info_file_3}")

Loading model_info_1 from file...
Loading model_info_2 from file...
Loading model_info_3 from file...


In [9]:
# NOTE: Dataset loading is now handled in cell 4 using get_dataset_info_path and get_dataset_csv_path
# The dataset is loaded as 'df' and the task is loaded as 'dataset_task'
# This cell is kept for backward compatibility but data/df should be used instead
data = df

In [10]:
# NOTE: Code fixing for llm_analysis_*.py files is now handled during generation
# in the analysis notebooks (with fix_code=True in MultiRunConfig).
# This cell has been removed to avoid redundant code fixing.

In [11]:
transform_functions_1 = {}
transform_functions_2 = {}
transform_functions_3 = {}
model_functions_1 = {}
model_functions_2 = {}
model_functions_3 = {}

# ----- get the transform and model functions for the first input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_1_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- get the transform and model functions for the second input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model

# ----- get the transform and model functions for the third input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_3_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_3[i] = module.transform
    model_functions_3[i] = module.model

In [12]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 1-{i}] Failed with error: {e}")
        transformed_datasets_1[i] = None

transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 2-{i}] Failed with error: {e}")
        transformed_datasets_2[i] = None
        
transformed_datasets_3 = {}
for i, transform_func in transform_functions_3.items():
    try:
        transformed_datasets_3[i] = transform_func(data.copy())
        print(f"[Transform 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 3-{i}] Failed with error: {e}")
        transformed_datasets_3[i] = None


model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] Failed with error: {e}")
        model_results_1[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] Failed with error: {e}")
        model_results_2[i] = None
        
model_results_3 = {}
for i, model_func in model_functions_3.items():
    try:
        if transformed_datasets_3[i] is None:
            print(f"[Model 3-{i}] Skipping — transform failed.")
            continue

        model_results_3[i] = model_func(transformed_datasets_3[i].copy())
        print(f"[Model 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 3-{i}] Failed with error: {e}")
        model_results_3[i] = None

[Transform 1-0] Completed successfully.
[Transform 1-1] Completed successfully.
[Transform 1-2] Completed successfully.
[Transform 2-0] Completed successfully.
[Transform 2-1] Completed successfully.
[Transform 2-2] Completed successfully.
[Transform 3-0] Completed successfully.
[Transform 3-1] Completed successfully.
[Transform 3-2] Completed successfully.
[Model 1-0] Completed successfully.
[Model 1-1] Completed successfully.


/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid 

[Model 1-2] Completed successfully.
[Model 2-0] Completed successfully.
[Model 2-1] Completed successfully.


/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparation

[Model 2-2] Completed successfully.
OLS on log(fish_per_hour) summary:
                            OLS Regression Results                            
Dep. Variable:      log_fish_per_hour   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.499
Method:                 Least Squares   F-statistic:                     72.48
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           1.93e-40
Time:                        09:27:03   Log-Likelihood:                -443.40
No. Observations:                 250   AIC:                             896.8
Df Residuals:                     245   BIC:                             914.4
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------

/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


In [13]:
# NOTE: Dataset info is now loaded in cell 4
# Use dataset_task (from cell 4) instead of task
task = dataset_task

# Generate final answer code files (llm_answer_*.py) for each analysis
# These are used later when generating conclusions
for i in range(num_analyses_1):
    answer_file = os.path.join(os.path.abspath(analysis_subdir_path_1), f"llm_answer_{i}.py")
    if os.path.exists(answer_file):
        print(f"Answer file {answer_file} already exists, skipping...")
        continue
    
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']
    
    # Format variables as text for the function
    cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
    
    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    print(f"Generating answer file {answer_file} (this will make LLM API calls)...")
    write_final_answer_code(
        llm_provider,
        llm_model,
        [task],  # task should be a list
        cvars_text,
        model_code,
        os.path.abspath(analysis_subdir_path_1),
        i,
        model_output
    )

for i in range(num_analyses_2):
    answer_file = os.path.join(os.path.abspath(analysis_subdir_path_2), f"llm_answer_{i}.py")
    if os.path.exists(answer_file):
        print(f"Answer file {answer_file} already exists, skipping...")
        continue
    
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']
    
    # Format variables as text for the function
    cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
    
    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    print(f"Generating answer file {answer_file} (this will make LLM API calls)...")
    write_final_answer_code(
        llm_provider,
        llm_model,
        [task],  # task should be a list
        cvars_text,
        model_code,
        os.path.abspath(analysis_subdir_path_2),
        i,
        model_output
    )

for i in range(num_analyses_3):
    answer_file = os.path.join(os.path.abspath(analysis_subdir_path_3), f"llm_answer_{i}.py")
    if os.path.exists(answer_file):
        print(f"Answer file {answer_file} already exists, skipping...")
        continue
    
    independent_variable = features_3[i]['independent_variables']
    dependent_variable = features_3[i]['response_variables']
    
    # Format variables as text for the function
    cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
    
    model_code = multirun_analyses_3['analyses'][str(i)]['m_code']
    model_output = model_results_3[i]

    print(f"Generating answer file {answer_file} (this will make LLM API calls)...")
    write_final_answer_code(
        llm_provider,
        llm_model,
        [task],  # task should be a list
        cvars_text,
        model_code,
        os.path.abspath(analysis_subdir_path_3),
        i,
        model_output
    )


Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis1_output/llm_answer_0.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis1_output/llm_answer_1.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis1_output/llm_answer_2.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_0.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_1.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_2.py already exists, skipping...
Answer file /accounts/campus/austin.zane/stat-genie/examples/fea

In [14]:
answer_code_paths_1 = [join(analysis_subdir_path_1, f"llm_answer_{i}.py") for i in range(num_analyses_1)]
answer_code_paths_2 = [join(analysis_subdir_path_2, f"llm_answer_{i}.py") for i in range(num_analyses_2)]
answer_code_paths_3 = [join(analysis_subdir_path_3, f"llm_answer_{i}.py") for i in range(num_analyses_3)]

In [15]:
# check that code works
answer_code_paths = [answer_code_paths_1, answer_code_paths_2,
                     answer_code_paths_3]
model_results = [model_results_1, model_results_2,
                 model_results_3]
for i in range(len(answer_code_paths)):
    for j, answer_code_path in enumerate(answer_code_paths[i]):
        # get absolute path using relative path so that the helper function works correctly
        absolute_path = os.path.abspath(answer_code_path)
        # call helper function to ensure code correctness
        num_iterations = check_and_fix_code(f"llm_answer_{j}",
                                            absolute_path,
                                            "final_answer",
                                            llm_provider,
                                            llm_model,
                                            model_output=model_results[i][j],
                                            verbose=False)
        print(f"Answer {i} iteration {j} required {num_iterations} correction iterations.")

Answer 0 iteration 0 required 0 correction iterations.
Answer 0 iteration 1 required 0 correction iterations.
Answer 0 iteration 2 required 0 correction iterations.
Answer 1 iteration 0 required 0 correction iterations.
Answer 1 iteration 1 required 0 correction iterations.
Answer 1 iteration 2 required 0 correction iterations.
Answer 2 iteration 0 required 0 correction iterations.
Answer 2 iteration 1 required 0 correction iterations.
Answer 2 iteration 2 required 0 correction iterations.


/accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_1.py:73: RuntimeWarning: overflow encountered in exp
  rr_ci_high = float(np.exp(ci_high)) if ci_high is not None else None


In [16]:
# get final answer functions in dict
final_answer_functions_1 = {}
final_answer_functions_2 = {}
final_answer_functions_3 = {}

# ----- get the final answer functions for the first input group -----
for i, answer_code_path in enumerate(answer_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_1_{i}"] = module
    spec.loader.exec_module(module)
    
    final_answer_functions_1[i] = module.extract_final_answer

# ----- get the final answer functions for the second input group -----
for i, answer_code_path in enumerate(answer_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_2_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_2[i] = module.extract_final_answer
    
# ----- get the final answer functions for the third input group -----
for i, answer_code_path in enumerate(answer_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_3_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_3[i] = module.extract_final_answer


In [17]:
final_answers_1 = {}
for i, final_answer_func in final_answer_functions_1.items():
    try:
        model_output = deepcopy(model_results_1[i])
        final_answers_1[i] = final_answer_func(model_output)
        print(f"[Answer 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 1-{i}] Failed with error: {e}")
        final_answers_1[i] = None

final_answers_2 = {}
for i, final_answer_func in final_answer_functions_2.items():
    try:
        model_output = deepcopy(model_results_2[i])
        final_answers_2[i] = final_answer_func(model_output)
        print(f"[Answer 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 2-{i}] Failed with error: {e}")
        final_answers_2[i] = None
        
final_answers_3 = {}
for i, final_answer_func in final_answer_functions_3.items():
    try:
        model_output = deepcopy(model_results_3[i])
        final_answers_3[i] = final_answer_func(model_output)
        print(f"[Answer 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 3-{i}] Failed with error: {e}")
        final_answers_3[i] = None

[Answer 1-0] Completed successfully.
[Answer 1-1] Completed successfully.
[Answer 1-2] Completed successfully.
[Answer 2-0] Completed successfully.
[Answer 2-1] Completed successfully.
[Answer 2-2] Completed successfully.
[Answer 3-0] Completed successfully.
[Answer 3-1] Completed successfully.
[Answer 3-2] Completed successfully.


/accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_1.py:73: RuntimeWarning: overflow encountered in exp
  rr_ci_high = float(np.exp(ci_high)) if ci_high is not None else None


In [18]:
# Load conclusions from text files (if they exist) or generate them
conclusions_1 = {}

for i in range(num_analyses_1):
    # Try to read from text file first
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    if os.path.exists(conclusion_txt):
        with open(conclusion_txt, "r", encoding="utf-8") as f:
            conclusions_1[i] = f.read()
    else:
        # Generate conclusion if file doesn't exist
        print(f"Generating conclusion_1[{i}] (this will make LLM API calls)...")
        independent_variable = features_1[i]['independent_variables']
        dependent_variable = features_1[i]['response_variables']
        # Format variables as text for the function
        cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
        model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
        code_filename = f"llm_answer_{i}.py"
        code_path = os.path.abspath(os.path.join(analysis_subdir_path_1, code_filename))
        with open(code_path, "r", encoding="utf-8") as f:
            interpretation_code_str = f.read()
        interpretation_output = final_answers_1[i] if i in final_answers_1 else None
        conclusions_1[i] = make_conclusion(
            llm_provider,
            llm_model,
            [task],  # task should be a list
            cvars_text,
            model_code,
            interpretation_code_str,
            interpretation_output
        )
        # Save to file
        with open(conclusion_txt, "w", encoding="utf-8") as f:
            f.write(conclusions_1[i])
        print(f"Saved conclusion_1[{i}] to {conclusion_txt}")

conclusions_2 = {}

for i in range(num_analyses_2):
    # Try to read from text file first
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_2, f"final_conclusion_{i}.txt"))
    if os.path.exists(conclusion_txt):
        with open(conclusion_txt, "r", encoding="utf-8") as f:
            conclusions_2[i] = f.read()
    else:
        # Generate conclusion if file doesn't exist
        print(f"Generating conclusion_2[{i}] (this will make LLM API calls)...")
        independent_variable = features_2[i]['independent_variables']
        dependent_variable = features_2[i]['response_variables']
        # Format variables as text for the function
        cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
        model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
        code_filename = f"llm_answer_{i}.py"
        code_path = os.path.abspath(os.path.join(analysis_subdir_path_2, code_filename))
        with open(code_path, "r", encoding="utf-8") as f:
            interpretation_code_str = f.read()
        interpretation_output = final_answers_2[i] if i in final_answers_2 else None
        conclusions_2[i] = make_conclusion(
            llm_provider,
            llm_model,
            [task],  # task should be a list
            cvars_text,
            model_code,
            interpretation_code_str,
            interpretation_output
        )
        # Save to file
        with open(conclusion_txt, "w", encoding="utf-8") as f:
            f.write(conclusions_2[i])
        print(f"Saved conclusion_2[{i}] to {conclusion_txt}")
    
conclusions_3 = {}

for i in range(num_analyses_3):
    # Try to read from text file first
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_3, f"final_conclusion_{i}.txt"))
    if os.path.exists(conclusion_txt):
        with open(conclusion_txt, "r", encoding="utf-8") as f:
            conclusions_3[i] = f.read()
    else:
        # Generate conclusion if file doesn't exist
        print(f"Generating conclusion_3[{i}] (this will make LLM API calls)...")
        independent_variable = features_3[i]['independent_variables']
        dependent_variable = features_3[i]['response_variables']
        # Format variables as text for the function
        cvars_text = f"Independent variables: {independent_variable}\nDependent variables: {dependent_variable}"
        model_code = multirun_analyses_3['analyses'][str(i)]['m_code']
        code_filename = f"llm_answer_{i}.py"
        code_path = os.path.abspath(os.path.join(analysis_subdir_path_3, code_filename))
        with open(code_path, "r", encoding="utf-8") as f:
            interpretation_code_str = f.read()
        interpretation_output = final_answers_3[i] if i in final_answers_3 else None
        conclusions_3[i] = make_conclusion(
            llm_provider,
            llm_model,
            [task],  # task should be a list
            cvars_text,
            model_code,
            interpretation_code_str,
            interpretation_output
        )
        # Save to file
        with open(conclusion_txt, "w", encoding="utf-8") as f:
            f.write(conclusions_3[i])
        print(f"Saved conclusion_3[{i}] to {conclusion_txt}")

In [19]:
# Prepare data head for judge evaluation
data_head = df.head(10)

# Convert dictionaries to lists for run_judge_evaluation_pairwise
# The function expects lists, but format_features returns dicts with integer keys
features_1_list = [features_1[i] for i in range(num_analyses_1)]
features_2_list = [features_2[i] for i in range(num_analyses_2)]
features_3_list = [features_3[i] for i in range(num_analyses_3)]

model_info_1_list = [model_info_1[i] for i in range(num_analyses_1)]
model_info_2_list = [model_info_2[i] for i in range(num_analyses_2)]
model_info_3_list = [model_info_3[i] for i in range(num_analyses_3)]

conclusions_1_list = [conclusions_1[i] for i in range(num_analyses_1)]
conclusions_2_list = [conclusions_2[i] for i in range(num_analyses_2)]
conclusions_3_list = [conclusions_3[i] for i in range(num_analyses_3)]

In [20]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons
# NOTE: Using run_judge_evaluation_pairwise which uses three separate judges
# for variables, models, and conclusions, then combines the results


In [21]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [22]:
### begin with within-group performance
# Load or generate within-group judge results
within_group_file = join(analysis_subdir_path_1, "..", "within_group_judge_results.json")

if os.path.exists(within_group_file):
    print("Loading within_group judge results from file...")
    with open(within_group_file, "r", encoding="utf-8") as f:
        within_group_dict = json.load(f)
    # Convert string keys back to int keys for groups and tuple keys for pairs
    within_group = {}
    for group_str, comparisons in within_group_dict.items():
        group_id = int(group_str)
        within_group[group_id] = {}
        for pair_str, result in comparisons.items():
            # Convert "(i, j)" string back to tuple
            pair_tuple = tuple(map(int, pair_str.strip("()").split(", ")))
            within_group[group_id][pair_tuple] = result
    print(f"Loaded within_group results from {within_group_file}")
else:
    print("Generating within_group judge results (this will make LLM API calls)...")
    within_group = {1: {}, 2: {}, 3: {}}

    # Group 1: within-group comparisons
    results_1 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_1_list,
    features_1_list,
    model_info_1_list,
    model_info_1_list,
    conclusions_1_list,
    conclusions_1_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
    # Filter to only keep (i, j) where i < j
    for (i, j), result in results_1.items():
        if i < j:
            within_group[1][(i, j)] = result

    # Group 2: within-group comparisons
    results_2 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_2_list,
    features_2_list,
    model_info_2_list,
    model_info_2_list,
    conclusions_2_list,
    conclusions_2_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
    # Filter to only keep (i, j) where i < j
    for (i, j), result in results_2.items():
        if i < j:
            within_group[2][(i, j)] = result

    # Group 3: within-group comparisons
    results_3 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_3_list,
    features_3_list,
    model_info_3_list,
    model_info_3_list,
    conclusions_3_list,
    conclusions_3_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
# Filter to only keep (i, j) where i < j
    for (i, j), result in results_3.items():
        if i < j:
            within_group[3][(i, j)] = result
    
    # Save to file (convert int keys to strings and tuple keys to strings for JSON)
    within_group_dict = {}
    for group_id, comparisons in within_group.items():
        within_group_dict[str(group_id)] = {}
        for pair, result in comparisons.items():
            within_group_dict[str(group_id)][str(pair)] = result
    
    with open(within_group_file, "w", encoding="utf-8") as f:
        json.dump(within_group_dict, f, indent=2)
    print(f"Saved within_group results to {within_group_file}")

Loading within_group judge results from file...
Loaded within_group results from outputs/analysis1_output/../within_group_judge_results.json


In [23]:
### now do between-group performance
# Load or generate between-group judge results
between_group_file = join(analysis_subdir_path_1, "..", "between_group_judge_results.json")

if os.path.exists(between_group_file):
    print("Loading between_group judge results from file...")
    with open(between_group_file, "r", encoding="utf-8") as f:
        between_group_dict = json.load(f)
    # Convert string keys back to tuple keys for group pairs and pair tuples
    between_group = {}
    for group_pair_str, comparisons in between_group_dict.items():
        # Convert "(1, 2)" string back to tuple
        group_pair = tuple(map(int, group_pair_str.strip("()").split(", ")))
        between_group[group_pair] = {}
        for pair_str, result in comparisons.items():
            # Convert "(i, j)" string back to tuple
            pair_tuple = tuple(map(int, pair_str.strip("()").split(", ")))
            between_group[group_pair][pair_tuple] = result
    print(f"Loaded between_group results from {between_group_file}")
else:
    print("Generating between_group judge results (this will make LLM API calls)...")
    between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }

    # Between group 1 and 2
    results_1_2 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_1_list,
    features_2_list,
    model_info_1_list,
    model_info_2_list,
    conclusions_1_list,
    conclusions_2_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
    between_group[(1, 2)] = results_1_2

    # Between group 1 and 3
    results_1_3 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_1_list,
    features_3_list,
    model_info_1_list,
    model_info_3_list,
    conclusions_1_list,
    conclusions_3_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
    between_group[(1, 3)] = results_1_3

    # Between group 2 and 3
    results_2_3 = run_judge_evaluation_pairwise(
    dataset_task,
    data_head,
    features_2_list,
    features_3_list,
    model_info_2_list,
    model_info_3_list,
    conclusions_2_list,
    conclusions_3_list,
    llm_provider=llm_provider,
    llm_model=llm_model
)
    between_group[(2, 3)] = results_2_3
    
    # Save to file (convert tuple keys to strings for JSON)
    between_group_dict = {}
    for group_pair, comparisons in between_group.items():
        between_group_dict[str(group_pair)] = {}
        for pair, result in comparisons.items():
            between_group_dict[str(group_pair)][str(pair)] = result
    
    with open(between_group_file, "w", encoding="utf-8") as f:
        json.dump(between_group_dict, f, indent=2)
    print(f"Saved between_group results to {between_group_file}")

Loading between_group judge results from file...
Loaded between_group results from outputs/analysis1_output/../between_group_judge_results.json


In [24]:
within_group

{1: {(0, 1): {'Independent Variables Similarity Score': 5,
   'Control Variables Similarity Score': 4,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 5,
   'conclusions': 5,
   'overall_similarity': 4.8},
  (0, 2): {'Independent Variables Similarity Score': 5,
   'Control Variables Similarity Score': 5,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 5,
   'conclusions': 5,
   'overall_similarity': 5.0},
  (1, 2): {'Independent Variables Similarity Score': 4,
   'Control Variables Similarity Score': 5,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 5,
   'conclusions': 5,
   'overall_similarity': 4.8}},
 2: {(0, 1): {'Independent Variables Similarity Score': 5,
   'Control Variables Similarity Score': 4,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 4,
   'conclusions': 1,
   'overall_similarity': 3.8},
  (0, 2): {'Independent Variables Similarity Score': 5,
   'Control Variables

In [25]:
# Print conclusion pairs with similarity ratings
import textwrap

print("=" * 80)
print("CONCLUSION PAIRS WITH SIMILARITY RATINGS")
print("=" * 80)

# Helper function to get conclusion similarity score from results
def get_conclusion_score(result_dict):
    """Extract conclusion similarity score from result dictionary"""
    # Try different possible key names
    if 'conclusions' in result_dict:
        return result_dict['conclusions']
    elif 'Conclusion Similarity Score' in result_dict:
        return result_dict['Conclusion Similarity Score']
    else:
        return None

def format_conclusion(text, max_length=1000, width=75):
    """Format conclusion text with wrapping and truncation"""
    if len(text) > max_length:
        text = text[:max_length] + "\n... [truncated]"
    # Wrap text to specified width
    wrapped = textwrap.fill(text, width=width, break_long_words=False, break_on_hyphens=False)
    return wrapped

# Within-group comparisons
print("\n" + "=" * 80)
print("WITHIN-GROUP COMPARISONS")
print("=" * 80)

group_names = {1: "Group 1 (baseline)", 2: "Group 2 (anonymized)", 3: "Group 3 (shuffled)"}

for group_id in [1, 2, 3]:
    print(f"\n{group_names[group_id]}:")
    print("-" * 80)
    
    if group_id == 1:
        conclusions_dict = conclusions_1
    elif group_id == 2:
        conclusions_dict = conclusions_2
    else:
        conclusions_dict = conclusions_3
    
    for (i, j), result in sorted(within_group[group_id].items()):
        concl_score = get_conclusion_score(result)
        print(f"\nPair ({i}, {j}) - Conclusion Similarity: {concl_score}/5")
        print(f"\nConclusion {i}:")
        print("-" * 80)
        print(format_conclusion(conclusions_dict[i]))
        print(f"\nConclusion {j}:")
        print("-" * 80)
        print(format_conclusion(conclusions_dict[j]))
        print("\n" + "=" * 80)

# Between-group comparisons
print("\n" + "=" * 80)
print("BETWEEN-GROUP COMPARISONS")
print("=" * 80)

comparison_names = {
    (1, 2): "Group 1 (baseline) vs Group 2 (anonymized)",
    (1, 3): "Group 1 (baseline) vs Group 3 (shuffled)",
    (2, 3): "Group 2 (anonymized) vs Group 3 (shuffled)"
}

for pair in [(1, 2), (1, 3), (2, 3)]:
    print(f"\n{comparison_names[pair]}:")
    print("-" * 80)
    
    if pair[0] == 1:
        concl_dict_1 = conclusions_1
    elif pair[0] == 2:
        concl_dict_1 = conclusions_2
    else:
        concl_dict_1 = conclusions_3
        
    if pair[1] == 1:
        concl_dict_2 = conclusions_1
    elif pair[1] == 2:
        concl_dict_2 = conclusions_2
    else:
        concl_dict_2 = conclusions_3
    
    for (i, j), result in sorted(between_group[pair].items()):
        concl_score = get_conclusion_score(result)
        print(f"\nPair ({pair[0]}-{i}, {pair[1]}-{j}) - Conclusion Similarity: {concl_score}/5")
        print(f"\nConclusion {pair[0]}-{i}:")
        print("-" * 80)
        print(format_conclusion(concl_dict_1[i]))
        print(f"\nConclusion {pair[1]}-{j}:")
        print("-" * 80)
        print(format_conclusion(concl_dict_2[j]))
        print("\n" + "=" * 80)

print("\n" + "=" * 80)
print("END OF CONCLUSION COMPARISONS")
print("=" * 80)



CONCLUSION PAIRS WITH SIMILARITY RATINGS

WITHIN-GROUP COMPARISONS

Group 1 (baseline):
--------------------------------------------------------------------------------

Pair (0, 1) - Conclusion Similarity: 5/5

Conclusion 0:
--------------------------------------------------------------------------------
{   "answer": "Yes",   "justification": "The fitted model (Negative
Binomial) yields interpretable rate estimates: using livebait is associated
with a ~4.24× higher catch rate (95% CI 2.27–7.91, p<0.001) and larger
group_size increases rate (~2.19× per additional person, p<0.001). Camper
presence is not significant. Example predicted rates: avg group (3.21
people) ≈ 0.284 fish/hr without livebait vs ≈ 1.202 fish/hr with livebait."
}

Conclusion 1:
--------------------------------------------------------------------------------
{   "answer": "Yes",   "justification": "The fitted model (Negative
Binomial; Poisson dispersion ≈13.85) provides an estimate of fish/hour:
baseline = 0.284 fi

In [26]:
# Print features and modeling comparisons for pair (1-0, 3-0)
import json

print("=" * 80)
print("FEATURES AND MODELING COMPARISON: Group 1-0 vs Group 3-0")
print("=" * 80)

# Get the similarity scores for this specific pair
pair_key = (0, 0)  # (i, j) indices in the between_group[(1, 3)] dictionary
result = between_group[(1, 3)][pair_key]

# Extract similarity scores
indep_score = result.get('Independent Variables Similarity Score', result.get('independent_variables', 'N/A'))
control_score = result.get('Control Variables Similarity Score', result.get('control_variables', 'N/A'))
response_score = result.get('Response Variables Similarity Score', result.get('response_variables', 'N/A'))
model_score = result.get('Model Similarity Score', result.get('model_specification', 'N/A'))

print(f"\nSimilarity Scores:")
print(f"  Independent Variables: {indep_score}/5")
print(f"  Control Variables: {control_score}/5")
print(f"  Response Variables: {response_score}/5")
print(f"  Model Specification: {model_score}/5")

# Helper function to format feature/model info
def format_dict(data, indent=2, max_length=2000):
    """Format dictionary or list data with wrapping"""
    text = json.dumps(data, indent=indent)
    if len(text) > max_length:
        text = text[:max_length] + "\n... [truncated]"
    # Wrap long lines
    wrapped_lines = []
    for line in text.split('\n'):
        if len(line) > 75:
            wrapped_lines.extend(textwrap.wrap(line, width=75, break_long_words=False, 
                                              break_on_hyphens=False, subsequent_indent='  '))
        else:
            wrapped_lines.append(line)
    return '\n'.join(wrapped_lines)

# INDEPENDENT VARIABLES
print("\n" + "=" * 80)
print("INDEPENDENT VARIABLES")
print("=" * 80)
print(f"\nSimilarity Score: {indep_score}/5")
print("\nGroup 1-0 Independent Variables:")
print("-" * 80)
print(format_dict(features_1[0]['independent_variables']))
print("\nGroup 3-0 Independent Variables:")
print("-" * 80)
print(format_dict(features_3[0]['independent_variables']))

# CONTROL VARIABLES
print("\n" + "=" * 80)
print("CONTROL VARIABLES")
print("=" * 80)
print(f"\nSimilarity Score: {control_score}/5")
print("\nGroup 1-0 Control Variables:")
print("-" * 80)
print(format_dict(features_1[0].get('control_variables', [])))
print("\nGroup 3-0 Control Variables:")
print("-" * 80)
print(format_dict(features_3[0].get('control_variables', [])))

# RESPONSE VARIABLES
print("\n" + "=" * 80)
print("RESPONSE VARIABLES")
print("=" * 80)
print(f"\nSimilarity Score: {response_score}/5")
print("\nGroup 1-0 Response Variables:")
print("-" * 80)
print(format_dict(features_1[0]['response_variables']))
print("\nGroup 3-0 Response Variables:")
print("-" * 80)
print(format_dict(features_3[0]['response_variables']))

# MODEL SPECIFICATION
print("\n" + "=" * 80)
print("MODEL SPECIFICATION")
print("=" * 80)
print(f"\nSimilarity Score: {model_score}/5")
print("\nGroup 1-0 Model Info:")
print("-" * 80)
# Model info might be a string (JSON) or dict
if isinstance(model_info_1[0], str):
    try:
        model_dict = json.loads(model_info_1[0])
        print(format_dict(model_dict))
    except:
        print(format_conclusion(model_info_1[0], max_length=2000))
else:
    print(format_dict(model_info_1[0]))
print("\nGroup 3-0 Model Info:")
print("-" * 80)
if isinstance(model_info_3[0], str):
    try:
        model_dict = json.loads(model_info_3[0])
        print(format_dict(model_dict))
    except:
        print(format_conclusion(model_info_3[0], max_length=2000))
else:
    print(format_dict(model_info_3[0]))

# Also show the actual model code
print("\n" + "=" * 80)
print("MODEL CODE")
print("=" * 80)
print("\nGroup 1-0 Model Code:")
print("-" * 80)
model_code_1 = multirun_analyses_1['analyses']['0']['m_code']
print(format_conclusion(model_code_1, max_length=2000))
print("\nGroup 3-0 Model Code:")
print("-" * 80)
model_code_3 = multirun_analyses_3['analyses']['0']['m_code']
print(format_conclusion(model_code_3, max_length=2000))

print("\n" + "=" * 80)
print("END OF FEATURES AND MODELING COMPARISON")
print("=" * 80)


FEATURES AND MODELING COMPARISON: Group 1-0 vs Group 3-0

Similarity Scores:
  Independent Variables: 4/5
  Control Variables: 4/5
  Response Variables: 4/5
  Model Specification: 3/5

INDEPENDENT VARIABLES

Similarity Score: 4/5

Group 1-0 Independent Variables:
--------------------------------------------------------------------------------
[
  {
    "description": "Use of live bait by the group (binary: 1 = used
  livebait, 0 = did not). Hypothesized to increase catch rate (fish per
  hour).",
    "columns": [
      "livebait"
    ],
    "transform_code": [
      "df = df.dropna(subset=required)\ndf['livebait'] =
  df['livebait'].astype(int)"
    ]
  },
  {
    "description": "Whether the group had a camper (binary: 1 = camper
  present, 0 = no camper). May affect time/effort or experience and thus
  influence catch rate.",
    "columns": [
      "camper"
    ],
    "transform_code": [
      "df['camper'] = df['camper'].astype(int)"
    ]
  }
]

Group 3-0 Independent Variables:
----

In [27]:
between_group

{(1,
  2): {(0, 0): {'Independent Variables Similarity Score': 4,
   'Control Variables Similarity Score': 5,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 4,
   'conclusions': 5,
   'overall_similarity': 4.6}, (0,
   1): {'Independent Variables Similarity Score': 4, 'Control Variables Similarity Score': 4, 'Response Variables Similarity Score': 5, 'Model Similarity Score': 4, 'conclusions': 1, 'overall_similarity': 3.6}, (0,
   2): {'Independent Variables Similarity Score': 5,
   'Control Variables Similarity Score': 5,
   'Response Variables Similarity Score': 5,
   'Model Similarity Score': 4,
   'conclusions': 4,
   'overall_similarity': 4.6}, (1,
   0): {'Independent Variables Similarity Score': 5, 'Control Variables Similarity Score': 4, 'Response Variables Similarity Score': 5, 'Model Similarity Score': 4, 'conclusions': 5, 'overall_similarity': 4.6}, (1,
   1): {'Independent Variables Similarity Score': 4,
   'Control Variables Similarity Score': 5,


In [28]:
# get average similarity score for each subcategory within each group
# Map the judge output keys to the expected keys
key_mapping = {
    "Independent Variables Similarity Score": "independent_variables",
    "Control Variables Similarity Score": "control_variables",
    "Response Variables Similarity Score": "response_variables",
    "Model Similarity Score": "model_specification",
    "conclusions": "conclusions",
    "overall_similarity": "overall_similarity"
}

average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for key, value in scores.items():
            # Map the key if it exists in the mapping, otherwise use the key as-is
            mapped_key = key_mapping.get(key, key)
            if mapped_key in category_sums:
                category_sums[mapped_key] += value
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [29]:
# show average within group rounded to nearest tenth
average_within_group

{1: {'independent_variables': 4.666666666666667,
  'control_variables': 4.666666666666667,
  'response_variables': 5.0,
  'model_specification': 5.0,
  'conclusions': 5.0,
  'overall_similarity': 4.866666666666667},
 2: {'independent_variables': 5.0,
  'control_variables': 3.3333333333333335,
  'response_variables': 5.0,
  'model_specification': 4.0,
  'conclusions': 2.0,
  'overall_similarity': 3.866666666666666},
 3: {'independent_variables': 4.333333333333333,
  'control_variables': 4.666666666666667,
  'response_variables': 4.666666666666667,
  'model_specification': 3.3333333333333335,
  'conclusions': 4.666666666666667,
  'overall_similarity': 4.333333333333333}}

In [30]:
# get average similarity score for each subcategory between each group
# Use the same key mapping as for within-group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for key, value in scores.items():
            # Map the key if it exists in the mapping, otherwise use the key as-is
            mapped_key = key_mapping.get(key, key)
            if mapped_key in category_sums:
                category_sums[mapped_key] += value
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [31]:
# Check for comparisons with similarity scores < 3 for independent, control, or response variables
print("=" * 80)
print("Checking for similarity scores < 3")
print("=" * 80)

# Score keys to check
score_keys = [
    'Independent Variables Similarity Score',
    'Control Variables Similarity Score',
    'Response Variables Similarity Score'
]

low_scores_found = []

# Check within-group comparisons
print("\nWITHIN-GROUP COMPARISONS:")
print("-" * 80)
for group_id, comparisons in within_group.items():
    for pair, scores in comparisons.items():
        for score_key in score_keys:
            score = scores.get(score_key)
            if score is not None and score < 3:
                low_scores_found.append({
                    'type': 'within_group',
                    'group': group_id,
                    'pair': pair,
                    'score_type': score_key,
                    'score': score
                })
                print(f"Group {group_id}, Pair {pair}: {score_key} = {score}")

# Check between-group comparisons
print("\nBETWEEN-GROUP COMPARISONS:")
print("-" * 80)
for group_pair, comparisons in between_group.items():
    for pair, scores in comparisons.items():
        for score_key in score_keys:
            score = scores.get(score_key)
            if score is not None and score < 3:
                low_scores_found.append({
                    'type': 'between_group',
                    'groups': group_pair,
                    'pair': pair,
                    'score_type': score_key,
                    'score': score
                })
                print(f"Groups {group_pair}, Pair {pair}: {score_key} = {score}")

# Summary
print("\n" + "=" * 80)
print("SUMMARY:")
print("-" * 80)
if low_scores_found:
    print(f"Found {len(low_scores_found)} comparison(s) with similarity scores < 3:")
    for item in low_scores_found:
        if item['type'] == 'within_group':
            print(f"  - Within Group {item['group']}, Pair {item['pair']}: {item['score_type']} = {item['score']}")
        else:
            print(f"  - Between Groups {item['groups']}, Pair {item['pair']}: {item['score_type']} = {item['score']}")
else:
    print("No comparisons found with similarity scores < 3 for independent, control, or response variables.")
print("=" * 80)


Checking for similarity scores < 3

WITHIN-GROUP COMPARISONS:
--------------------------------------------------------------------------------
Group 2, Pair (1, 2): Control Variables Similarity Score = 1

BETWEEN-GROUP COMPARISONS:
--------------------------------------------------------------------------------
Groups (1, 2), Pair (1, 2): Control Variables Similarity Score = 2
Groups (1, 3), Pair (2, 1): Control Variables Similarity Score = 1
Groups (2, 3), Pair (2, 0): Control Variables Similarity Score = 1
Groups (2, 3), Pair (2, 1): Control Variables Similarity Score = 1

SUMMARY:
--------------------------------------------------------------------------------
Found 5 comparison(s) with similarity scores < 3:
  - Within Group 2, Pair (1, 2): Control Variables Similarity Score = 1
  - Between Groups (1, 2), Pair (1, 2): Control Variables Similarity Score = 2
  - Between Groups (1, 3), Pair (2, 1): Control Variables Similarity Score = 1
  - Between Groups (2, 3), Pair (2, 0): Control 

In [32]:
average_between_group

{(1, 2): {'independent_variables': 4.333333333333333,
  'control_variables': 4.111111111111111,
  'response_variables': 5.0,
  'model_specification': 4.0,
  'conclusions': 3.3333333333333335,
  'overall_similarity': 4.155555555555555},
 (1, 3): {'independent_variables': 4.0,
  'control_variables': 3.7777777777777777,
  'response_variables': 4.333333333333333,
  'model_specification': 3.3333333333333335,
  'conclusions': 4.555555555555555,
  'overall_similarity': 3.999999999999999},
 (2, 3): {'independent_variables': 4.111111111111111,
  'control_variables': 3.5555555555555554,
  'response_variables': 4.666666666666667,
  'model_specification': 4.0,
  'conclusions': 3.111111111111111,
  'overall_similarity': 3.888888888888889}}

In [33]:
# summary of results
print("=" * 80)
print("pairwise similarity results")
print("=" * 80)

print("\nwithin-group consistency")
print("-" * 80)
for group_id, scores in average_within_group.items():
    group_name = {1: "baseline (no perturbation)", 2: "anonymized names", 3: "shuffled names"}[group_id]
    overall = scores['overall_similarity']
    status = "good" if overall >= 3.5 else "moderate" if overall >= 3.0 else "low"
    print(f"\ngroup {group_id} ({group_name}):")
    print(f"  overall similarity: {overall:.2f} ({status})")
    print(f"  - independent variables: {scores['independent_variables']:.2f}")
    print(f"  - control variables: {scores['control_variables']:.2f}")
    print(f"  - response variables: {scores['response_variables']:.2f}")
    print(f"  - model specification: {scores['model_specification']:.2f}")
    print(f"  - conclusions: {scores['conclusions']:.2f}")

print("\n\nbetween-group comparisons")
print("-" * 80)
comparison_names = {
    (1, 2): "baseline vs anonymized",
    (1, 3): "baseline vs shuffled",
    (2, 3): "anonymized vs shuffled"
}
for pair, scores in average_between_group.items():
    overall = scores['overall_similarity']
    baseline_score = average_within_group[1]['overall_similarity']
    drop = baseline_score - overall if pair[0] == 1 else None
    
    if pair == (1, 2):
        status = "robust" if overall >= 3.8 else "some degradation" if overall >= 3.0 else "poor"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
        print(f"  tests whether the model can work without semantic name cues")
    elif pair == (1, 3):
        status = "robust" if overall >= 3.5 else "affected" if overall >= 3.0 else "heavily affected"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
        if drop:
            print(f"  drop from baseline: {drop:.2f} points")
        print(f"  tests whether the model is fooled by misleading names")
    else:
        status = "similar" if overall >= 3.0 else "different"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
    
    print(f"  - independent variables: {scores['independent_variables']:.2f}")
    print(f"  - control variables: {scores['control_variables']:.2f}")
    print(f"  - response variables: {scores['response_variables']:.2f}")
    print(f"  - model specification: {scores['model_specification']:.2f}")
    print(f"  - conclusions: {scores['conclusions']:.2f}")

print("\n\nnotes")
print("-" * 80)
baseline_consistency = average_within_group[1]['overall_similarity']
anon_vs_baseline = average_between_group[(1, 2)]['overall_similarity']
shuffled_vs_baseline = average_between_group[(1, 3)]['overall_similarity']
shuffled_consistency = average_within_group[3]['overall_similarity']

notes = []
if baseline_consistency < 3.0:
    notes.append("high variability even in baseline condition")
if anon_vs_baseline >= 3.8:
    notes.append("model is robust to anonymization - works well without semantic names")
elif anon_vs_baseline < 3.0:
    notes.append("model struggles without semantic name cues")
if shuffled_vs_baseline < 3.0:
    notes.append("model is heavily affected by shuffled/misleading names")
    if shuffled_consistency < 3.0:
        notes.append("  shuffled condition also shows low internal consistency")
elif shuffled_vs_baseline >= 3.5:
    notes.append("model resists misleading names effectively")

if average_between_group[(1, 3)]['conclusions'] < average_between_group[(1, 3)]['model_specification']:
    notes.append("conclusions more affected than model specifications - interpretation instability")

if not notes:
    notes.append("overall: model shows reasonable robustness to feature name perturbations")

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

print("\n" + "=" * 80)


pairwise similarity results

within-group consistency
--------------------------------------------------------------------------------

group 1 (baseline (no perturbation)):
  overall similarity: 4.87 (good)
  - independent variables: 4.67
  - control variables: 4.67
  - response variables: 5.00
  - model specification: 5.00
  - conclusions: 5.00

group 2 (anonymized names):
  overall similarity: 3.87 (good)
  - independent variables: 5.00
  - control variables: 3.33
  - response variables: 5.00
  - model specification: 4.00
  - conclusions: 2.00

group 3 (shuffled names):
  overall similarity: 4.33 (good)
  - independent variables: 4.33
  - control variables: 4.67
  - response variables: 4.67
  - model specification: 3.33
  - conclusions: 4.67


between-group comparisons
--------------------------------------------------------------------------------

(1, 2) (baseline vs anonymized):
  overall similarity: 4.16 (robust)
  tests whether the model can work without semantic name cues
  - 

In [34]:
# Check for comparisons with similarity scores < 3 for independent, control, or response variables
print("=" * 80)
print("Checking for similarity scores < 3")
print("=" * 80)

# Score keys to check
score_keys = [
    'Independent Variables Similarity Score',
    'Control Variables Similarity Score',
    'Response Variables Similarity Score'
]

low_scores_found = []

# Check within-group comparisons
print("\nWITHIN-GROUP COMPARISONS:")
print("-" * 80)
for group_id, comparisons in within_group.items():
    for pair, scores in comparisons.items():
        for score_key in score_keys:
            score = scores.get(score_key)
            if score is not None and score < 3:
                low_scores_found.append({
                    'type': 'within_group',
                    'group': group_id,
                    'pair': pair,
                    'score_type': score_key,
                    'score': score
                })
                print(f"Group {group_id}, Pair {pair}: {score_key} = {score}")

# Check between-group comparisons
print("\nBETWEEN-GROUP COMPARISONS:")
print("-" * 80)
for group_pair, comparisons in between_group.items():
    for pair, scores in comparisons.items():
        for score_key in score_keys:
            score = scores.get(score_key)
            if score is not None and score < 3:
                low_scores_found.append({
                    'type': 'between_group',
                    'groups': group_pair,
                    'pair': pair,
                    'score_type': score_key,
                    'score': score
                })
                print(f"Groups {group_pair}, Pair {pair}: {score_key} = {score}")

# Summary
print("\n" + "=" * 80)
print("SUMMARY:")
print("-" * 80)
if low_scores_found:
    print(f"Found {len(low_scores_found)} comparison(s) with similarity scores < 3:")
    for item in low_scores_found:
        if item['type'] == 'within_group':
            print(f"  - Within Group {item['group']}, Pair {item['pair']}: {item['score_type']} = {item['score']}")
        else:
            print(f"  - Between Groups {item['groups']}, Pair {item['pair']}: {item['score_type']} = {item['score']}")
else:
    print("No comparisons found with similarity scores < 3 for independent, control, or response variables.")
print("=" * 80)


Checking for similarity scores < 3

WITHIN-GROUP COMPARISONS:
--------------------------------------------------------------------------------
Group 2, Pair (1, 2): Control Variables Similarity Score = 1

BETWEEN-GROUP COMPARISONS:
--------------------------------------------------------------------------------
Groups (1, 2), Pair (1, 2): Control Variables Similarity Score = 2
Groups (1, 3), Pair (2, 1): Control Variables Similarity Score = 1
Groups (2, 3), Pair (2, 0): Control Variables Similarity Score = 1
Groups (2, 3), Pair (2, 1): Control Variables Similarity Score = 1

SUMMARY:
--------------------------------------------------------------------------------
Found 5 comparison(s) with similarity scores < 3:
  - Within Group 2, Pair (1, 2): Control Variables Similarity Score = 1
  - Between Groups (1, 2), Pair (1, 2): Control Variables Similarity Score = 2
  - Between Groups (1, 3), Pair (2, 1): Control Variables Similarity Score = 1
  - Between Groups (2, 3), Pair (2, 0): Control 

In [35]:
# Display all similarity scores for each pair of analyses
print("=" * 100)
print("SIMILARITY SCORES FOR ALL PAIRS")
print("=" * 100)

# Score keys to display
score_keys = [
    'Independent Variables Similarity Score',
    'Control Variables Similarity Score',
    'Response Variables Similarity Score',
    'Model Similarity Score',
    'conclusions',
    'overall_similarity'
]

# Helper function to format scores
def format_scores(scores_dict, score_keys):
    """Format scores into a readable string"""
    formatted = []
    for key in score_keys:
        value = scores_dict.get(key, 'N/A')
        if value != 'N/A':
            # Shorten key names for display
            short_key = key.replace(' Similarity Score', '').replace(' Variables', ' Vars')
            formatted.append(f"{short_key}: {value}/5" if key != 'overall_similarity' else f"{short_key}: {value:.2f}")
    return " | ".join(formatted)

# Within-group comparisons
print("\n" + "=" * 100)
print("WITHIN-GROUP COMPARISONS")
print("=" * 100)

for group_id in sorted(within_group.keys()):
    print(f"\n{'─' * 100}")
    print(f"GROUP {group_id} (within-group comparisons)")
    print(f"{'─' * 100}")
    
    if not within_group[group_id]:
        print("  No comparisons available")
        continue
    
    for pair in sorted(within_group[group_id].keys()):
        scores = within_group[group_id][pair]
        print(f"\n  Pair {pair[0]} vs {pair[1]}:")
        print(f"    {format_scores(scores, score_keys)}")

# Between-group comparisons
print("\n" + "=" * 100)
print("BETWEEN-GROUP COMPARISONS")
print("=" * 100)

for group_pair in sorted(between_group.keys()):
    print(f"\n{'─' * 100}")
    print(f"GROUPS {group_pair[0]} vs {group_pair[1]} (between-group comparisons)")
    print(f"{'─' * 100}")
    
    if not between_group[group_pair]:
        print("  No comparisons available")
        continue
    
    for pair in sorted(between_group[group_pair].keys()):
        scores = between_group[group_pair][pair]
        print(f"\n  Pair {pair[0]} vs {pair[1]}:")
        print(f"    {format_scores(scores, score_keys)}")

print("\n" + "=" * 100)
print("END OF SIMILARITY SCORES")
print("=" * 100)


SIMILARITY SCORES FOR ALL PAIRS

WITHIN-GROUP COMPARISONS

────────────────────────────────────────────────────────────────────────────────────────────────────
GROUP 1 (within-group comparisons)
────────────────────────────────────────────────────────────────────────────────────────────────────

  Pair 0 vs 1:
    Independent Vars: 5/5 | Control Vars: 4/5 | Response Vars: 5/5 | Model: 5/5 | conclusions: 5/5 | overall_similarity: 4.80

  Pair 0 vs 2:
    Independent Vars: 5/5 | Control Vars: 5/5 | Response Vars: 5/5 | Model: 5/5 | conclusions: 5/5 | overall_similarity: 5.00

  Pair 1 vs 2:
    Independent Vars: 4/5 | Control Vars: 5/5 | Response Vars: 5/5 | Model: 5/5 | conclusions: 5/5 | overall_similarity: 4.80

────────────────────────────────────────────────────────────────────────────────────────────────────
GROUP 2 (within-group comparisons)
────────────────────────────────────────────────────────────────────────────────────────────────────

  Pair 0 vs 1:
    Independent Vars: 5/

In [44]:
# Display actual content compared for a specific pair
# Set these variables to specify which pair to examine:

# For within-group comparison:
# comparison_type = "within"  # or "between"
# group_id = 2  # For within-group: 1, 2, or 3
# pair = (1, 2)  # The pair indices (i, j)

# For between-group comparison (set comparison_type = "between"):
comparison_type = "between"
group_pair = (1, 2)  # The two groups being compared
pair = (2, 1)  # The pair indices (i, j)

print("=" * 100)
print("COMPARISON CONTENT FOR SPECIFIED PAIR")
print("=" * 100)

# Helper functions to format content in JSON-like format
def format_independent_variables(indep_vars):
    """Format independent variables as a single dict"""
    if isinstance(indep_vars, list) and len(indep_vars) > 0:
        # Take first independent variable (or combine if multiple)
        first = indep_vars[0]
        if isinstance(first, dict):
            result = {
                'description': first.get('description', ''),
                'columns': first.get('columns', [])
            }
            return json.dumps(result, indent=2)
    elif isinstance(indep_vars, dict):
        result = {
            'description': indep_vars.get('description', ''),
            'columns': indep_vars.get('columns', [])
        }
        return json.dumps(result, indent=2)
    return json.dumps({}, indent=2)

def format_control_variables(control_vars):
    """Format control variables as a list of dicts"""
    if not isinstance(control_vars, list):
        return json.dumps([], indent=2)
    
    formatted_list = []
    for var in control_vars:
        if isinstance(var, dict):
            formatted_list.append({
                'columns': var.get('columns', []),
                'description': var.get('description', ''),
                'is_moderator': var.get('is_moderator', False),
                'moderator_on': var.get('moderator_on', None)
            })
    
    return json.dumps(formatted_list, indent=2)

def format_response_variables(response_vars):
    """Format response variables as a dict"""
    if isinstance(response_vars, dict):
        result = {
            'description': response_vars.get('description', ''),
            'columns': response_vars.get('columns', [])
        }
        return json.dumps(result, indent=2)
    return json.dumps({}, indent=2)

def format_model_info(model_info):
    """Format model info - handle both dict and string"""
    if isinstance(model_info, str):
        try:
            # Try to parse as JSON
            parsed = json.loads(model_info)
            return json.dumps(parsed, indent=2)
        except:
            # If not JSON, return as-is (might be plain text)
            return model_info
    elif isinstance(model_info, dict):
        return json.dumps(model_info, indent=2)
    else:
        return str(model_info)

def format_conclusion(conclusion):
    """Format conclusion - handle both dict and string"""
    if isinstance(conclusion, str):
        try:
            # Try to parse as JSON
            parsed = json.loads(conclusion)
            return json.dumps(parsed, indent=2)
        except:
            # If not JSON, wrap in a dict
            return json.dumps({"text": conclusion}, indent=2)
    elif isinstance(conclusion, dict):
        return json.dumps(conclusion, indent=2)
    else:
        return json.dumps({"text": str(conclusion)}, indent=2)

# Get the data based on comparison type
if comparison_type == "within":
    print(f"\nWITHIN-GROUP COMPARISON: Group {group_id}, Pair {pair[0]} vs {pair[1]}")
    print("=" * 100)
    
    # Get similarity scores
    if pair in within_group.get(group_id, {}):
        scores = within_group[group_id][pair]
        print("\nSIMILARITY SCORES:")
        print("-" * 100)
        for key in ['Independent Variables Similarity Score', 'Control Variables Similarity Score', 
                    'Response Variables Similarity Score', 'Model Similarity Score', 'conclusions', 'overall_similarity']:
            value = scores.get(key, 'N/A')
            if value != 'N/A':
                print(f"  {key}: {value}")
    else:
        print(f"\nWARNING: Pair {pair} not found in within_group[{group_id}]")
        print("Available pairs:", list(within_group.get(group_id, {}).keys()))
    
    # Get features
    features_dict = {1: features_1, 2: features_2, 3: features_3}
    model_info_dict = {1: model_info_1, 2: model_info_2, 3: model_info_3}
    conclusions_dict = {1: conclusions_1, 2: conclusions_2, 3: conclusions_3}
    
    i, j = pair
    feat_i = features_dict[group_id].get(i, {})
    feat_j = features_dict[group_id].get(j, {})
    model_i = model_info_dict[group_id].get(i, "")
    model_j = model_info_dict[group_id].get(j, "")
    concl_i = conclusions_dict[group_id].get(i, "")
    concl_j = conclusions_dict[group_id].get(j, "")
    
    print(f"\n{'=' * 100}")
    print(f"ANALYSIS {i} (Group {group_id})")
    print("=" * 100)
    
    print("\nINDEPENDENT VARIABLES:")
    print("-" * 100)
    print(format_independent_variables(feat_i.get('independent_variables', {})))
    
    print("\nCONTROL VARIABLES:")
    print("-" * 100)
    print(format_control_variables(feat_i.get('control_variables', [])))
    
    print("\nRESPONSE VARIABLES:")
    print("-" * 100)
    print(format_response_variables(feat_i.get('response_variables', {})))
    
    print("\nMODEL SPECIFICATION:")
    print("-" * 100)
    print(format_model_info(model_i))
    
    print("\nCONCLUSION:")
    print("-" * 100)
    print(format_conclusion(concl_i))
    
    print(f"\n{'=' * 100}")
    print(f"ANALYSIS {j} (Group {group_id})")
    print("=" * 100)
    
    print("\nINDEPENDENT VARIABLES:")
    print("-" * 100)
    print(format_independent_variables(feat_j.get('independent_variables', {})))
    
    print("\nCONTROL VARIABLES:")
    print("-" * 100)
    print(format_control_variables(feat_j.get('control_variables', [])))
    
    print("\nRESPONSE VARIABLES:")
    print("-" * 100)
    print(format_response_variables(feat_j.get('response_variables', {})))
    
    print("\nMODEL SPECIFICATION:")
    print("-" * 100)
    print(format_model_info(model_j))
    
    print("\nCONCLUSION:")
    print("-" * 100)
    print(format_conclusion(concl_j))

elif comparison_type == "between":
    # Uncomment and set these for between-group comparison:
    # group_pair = (1, 3)
    # pair = (0, 0)
    
    if 'group_pair' not in locals():
        print("\nERROR: Please set 'group_pair' variable for between-group comparison")
        print("Example: group_pair = (1, 3)")
    else:
        print(f"\nBETWEEN-GROUP COMPARISON: Groups {group_pair[0]} vs {group_pair[1]}, Pair {pair[0]} vs {pair[1]}")
        print("=" * 100)
        
        # Get similarity scores
        if group_pair in between_group and pair in between_group[group_pair]:
            scores = between_group[group_pair][pair]
            print("\nSIMILARITY SCORES:")
            print("-" * 100)
            for key in ['Independent Variables Similarity Score', 'Control Variables Similarity Score', 
                        'Response Variables Similarity Score', 'Model Similarity Score', 'conclusions', 'overall_similarity']:
                value = scores.get(key, 'N/A')
                if value != 'N/A':
                    print(f"  {key}: {value}")
        else:
            print(f"\nWARNING: Pair {pair} not found in between_group[{group_pair}]")
            if group_pair in between_group:
                print("Available pairs:", list(between_group[group_pair].keys()))
        
        # Get features
        features_dict = {1: features_1, 2: features_2, 3: features_3}
        model_info_dict = {1: model_info_1, 2: model_info_2, 3: model_info_3}
        conclusions_dict = {1: conclusions_1, 2: conclusions_2, 3: conclusions_3}
        
        i, j = pair
        g1, g2 = group_pair
        
        feat_i = features_dict[g1].get(i, {})
        feat_j = features_dict[g2].get(j, {})
        model_i = model_info_dict[g1].get(i, "")
        model_j = model_info_dict[g2].get(j, "")
        concl_i = conclusions_dict[g1].get(i, "")
        concl_j = conclusions_dict[g2].get(j, "")
        
        print(f"\n{'=' * 100}")
        print(f"ANALYSIS {i} (Group {g1})")
        print("=" * 100)
        
        print("\nINDEPENDENT VARIABLES:")
        print("-" * 100)
        print(format_independent_variables(feat_i.get('independent_variables', {})))
        
        print("\nCONTROL VARIABLES:")
        print("-" * 100)
        print(format_control_variables(feat_i.get('control_variables', [])))
        
        print("\nRESPONSE VARIABLES:")
        print("-" * 100)
        print(format_response_variables(feat_i.get('response_variables', {})))
        
        print("\nMODEL SPECIFICATION:")
        print("-" * 100)
        print(format_model_info(model_i))
        
        print("\nCONCLUSION:")
        print("-" * 100)
        print(format_conclusion(concl_i))
        
        print(f"\n{'=' * 100}")
        print(f"ANALYSIS {j} (Group {g2})")
        print("=" * 100)
        
        print("\nINDEPENDENT VARIABLES:")
        print("-" * 100)
        print(format_independent_variables(feat_j.get('independent_variables', {})))
        
        print("\nCONTROL VARIABLES:")
        print("-" * 100)
        print(format_control_variables(feat_j.get('control_variables', [])))
        
        print("\nRESPONSE VARIABLES:")
        print("-" * 100)
        print(format_response_variables(feat_j.get('response_variables', {})))
        
        print("\nMODEL SPECIFICATION:")
        print("-" * 100)
        print(format_model_info(model_j))
        
        print("\nCONCLUSION:")
        print("-" * 100)
        print(format_conclusion(concl_j))

print("\n" + "=" * 100)


COMPARISON CONTENT FOR SPECIFIED PAIR

BETWEEN-GROUP COMPARISON: Groups 1 vs 2, Pair 2 vs 1

SIMILARITY SCORES:
----------------------------------------------------------------------------------------------------
  Independent Variables Similarity Score: 4
  Control Variables Similarity Score: 3
  Response Variables Similarity Score: 5
  Model Similarity Score: 4
  conclusions: 1
  overall_similarity: 3.4

ANALYSIS 2 (Group 1)

INDEPENDENT VARIABLES:
----------------------------------------------------------------------------------------------------
{
  "description": "Whether the group used live bait (binary indicator: 1 = used live bait, 0 = did not). Primary predictor hypothesized to increase catch rate.",
  "columns": [
    "livebait"
  ]
}

CONTROL VARIABLES:
----------------------------------------------------------------------------------------------------
[]

RESPONSE VARIABLES:
----------------------------------------------------------------------------------------------------

In [37]:
# Print the research question
print("Research Question:")
print("=" * 80)
print(dataset_task)
print("=" * 80)


Research Question:
How many fish on average do visitors takes per hour, when fishing? (i.e., What factors influence the number of fish caught by visitors to a national park and how can we estimate the rate of fish caught per hour?)
